In [1]:
from pathlib import Path

import pandas as pd


In [2]:
PROJECT_ROOT = Path.cwd().parent
DATA_DIR = PROJECT_ROOT / "data" / "raw"

TRAIN_PATH = DATA_DIR / "medical_tc_train.csv"
TEST_PATH = DATA_DIR / "medical_tc_test.csv"
LABELS_PATH = DATA_DIR / "medical_tc_labels.csv"

In [3]:
train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)
labels_df = pd.read_csv(LABELS_PATH)

In [4]:
train_df.head()

,condition_label,medical_abstract
0,5,Tissue changes around loose prostheses. A cani...
1,1,Neuropeptide Y and neuron-specific enolase lev...
2,2,"Sexually transmitted diseases of the colon, re..."
3,1,Lipolytic factors associated with murine and h...
4,3,Does carotid restenosis predict an increased r...


In [5]:
print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)

Train shape: (11550, 2)
Test shape: (2888, 2)


In [6]:
total_samples = len(train_df) + len(test_df)

print("Total samples:", total_samples)

Total samples: 14438


In [7]:
train_df.columns.tolist()

['condition_label', 'medical_abstract']

In [8]:
TEXT_COL = "medical_abstract"
TARGET_COL = "condition_label"

In [9]:
train_df[TARGET_COL].unique()

array([5, 1, 2, 3, 4])

In [10]:
train_df[TARGET_COL].nunique()

5

In [11]:
labels_df

,condition_label,condition_name
0,1,neoplasms
1,2,digestive system diseases
2,3,nervous system diseases
3,4,cardiovascular diseases
4,5,general pathological conditions


# Distribuição de classes

In [12]:
train_df[TARGET_COL].value_counts().sort_index()

condition_label
1    2530
2    1195
3    1540
4    2441
5    3844
Name: count, dtype: int64

In [13]:
train_df[TARGET_COL].value_counts(
    normalize=True
).sort_index() * 100

condition_label
1    21.904762
2    10.346320
3    13.333333
4    21.134199
5    33.281385
Name: proportion, dtype: float64

# Valores Ausentes

In [14]:
train_df.isna().sum()

condition_label     0
medical_abstract    0
dtype: int64

In [15]:
test_df.isna().sum()

condition_label     0
medical_abstract    0
dtype: int64

# Duplicatas

In [16]:
train_df.duplicated().sum()

np.int64(0)

In [17]:
train_df[TEXT_COL].duplicated().sum()

np.int64(2105)

In [18]:
test_df[TEXT_COL].duplicated().sum()

np.int64(118)

# Leakage entre train e test

In [19]:
train_texts = set(train_df[TEXT_COL])
test_texts = set(test_df[TEXT_COL])

overlap = train_texts.intersection(test_texts)

len(overlap)

988

In [20]:
overlap_df = (
    train_df[[TEXT_COL, TARGET_COL]]
    .merge(
        test_df[[TEXT_COL, TARGET_COL]],
        on=TEXT_COL,
        suffixes=("_train", "_test")
    )
)

overlap_df.head()

,medical_abstract,condition_label_train,condition_label_test
0,The management of postoperative chylous ascite...,2,5
1,Catheterization of coronary artery bypass graf...,4,5
2,Therapy of renal cell carcinoma with interleuk...,1,4
3,Primary intratemporal tumours of the facial ne...,3,1
4,Preventing colorectal cancer. Knowledgeable pa...,1,2


In [21]:
same_label = (
    overlap_df[f"{TARGET_COL}_train"]
    == overlap_df[f"{TARGET_COL}_test"]
)

same_label.value_counts()

False    1119
Name: count, dtype: int64

In [22]:
conflicting_labels = overlap_df[
    overlap_df[f"{TARGET_COL}_train"]
    != overlap_df[f"{TARGET_COL}_test"]
]

conflicting_labels.shape

(1119, 3)

In [23]:
conflicting_labels.head()

,medical_abstract,condition_label_train,condition_label_test
0,The management of postoperative chylous ascite...,2,5
1,Catheterization of coronary artery bypass graf...,4,5
2,Therapy of renal cell carcinoma with interleuk...,1,4
3,Primary intratemporal tumours of the facial ne...,3,1
4,Preventing colorectal cancer. Knowledgeable pa...,1,2


In [24]:
print(
    "Duplicados no train:",
    train_df[TEXT_COL].duplicated().sum()
)

print(
    "Duplicados no test:",
    test_df[TEXT_COL].duplicated().sum()
)

Duplicados no train: 2105
Duplicados no test: 118


In [25]:
all_df = pd.concat(
    [train_df, test_df],
    ignore_index=True
)

In [26]:
labels_per_text = (
    all_df
    .groupby(TEXT_COL)[TARGET_COL]
    .nunique()
)

In [27]:
labels_per_text.value_counts().sort_index()

condition_label
1    8298
2    2653
3     270
4       6
Name: count, dtype: int64

In [28]:
ambiguous_texts = labels_per_text[
    labels_per_text > 1
]

len(ambiguous_texts)

2929

In [29]:
ambiguous_examples = all_df[
    all_df[TEXT_COL].isin(ambiguous_texts.index)
].sort_values(TEXT_COL)

ambiguous_examples[
    [TARGET_COL, TEXT_COL]
].head(20)

,condition_label,medical_abstract
8424,5,'Locked-in syndrome' for 27 years following a ...
3987,3,'Locked-in syndrome' for 27 years following a ...
10457,1,'Locked-in syndrome' for 27 years following a ...
3090,5,(A)typical symptoms during single needle dialy...
10683,4,(A)typical symptoms during single needle dialy...
7960,2,(A)typical symptoms during single needle dialy...
14304,5,7th nerve palsy after extradural blood patch. ...
5969,3,7th nerve palsy after extradural blood patch. ...
1973,3,99TCm-HMPAO SPECT studies in traumatic intrace...
954,5,99TCm-HMPAO SPECT studies in traumatic intrace...


In [30]:
train_labels_per_text = (
    train_df
    .groupby(TEXT_COL)[TARGET_COL]
    .nunique()
)

train_labels_per_text.value_counts().sort_index()

condition_label
1    7489
2    1810
3     143
4       3
Name: count, dtype: int64

In [31]:
test_labels_per_text = (
    test_df
    .groupby(TEXT_COL)[TARGET_COL]
    .nunique()
)

test_labels_per_text.value_counts().sort_index()

condition_label
1    2657
2     108
3       5
Name: count, dtype: int64

In [32]:
len(ambiguous_texts)

2929

In [33]:
unambiguous_df = all_df[
    ~all_df[TEXT_COL].isin(ambiguous_texts.index)
].copy()

In [34]:
unambiguous_df = unambiguous_df.drop_duplicates(
    subset=[TEXT_COL]
)

In [35]:
unambiguous_df.shape

(8298, 2)

In [36]:
unambiguous_df[TARGET_COL].value_counts().sort_index()

condition_label
1    2195
2     699
3    1049
4    1961
5    2394
Name: count, dtype: int64

In [37]:
(
    unambiguous_df[TARGET_COL]
    .value_counts(normalize=True)
    .sort_index()
    * 100
)

condition_label
1    26.452157
2     8.423717
3    12.641600
4    23.632201
5    28.850325
Name: proportion, dtype: float64

In [38]:
all_df[TARGET_COL].value_counts().sort_index()

condition_label
1    3163
2    1494
3    1925
4    3051
5    4805
Name: count, dtype: int64